In [ ]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
!pip install "datasets<4.0.0" transformers sae-lens transformer-lens accelerate torch

INFO: pip is looking at multiple versions of transformer-lens to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.9/290.9 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [6]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import gc
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

MODELS_CONFIG = [
    {
        "name": "Qwen 1.5B",
        "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "sae_release": "DGurgurov/DeepSeek-R1-Distill-Qwen-1.5B-sae",
        "sae_ids": [
            "blocks.1.hook_resid_pre"
        ]
    },
    {
        "name": "GPT-2 Small",
        "model_id": "gpt2",
        "sae_release": "gpt2-small-res-jb",
        "sae_ids": [
            "blocks.0.hook_resid_pre",
            "blocks.6.hook_resid_pre",
            "blocks.11.hook_resid_pre"
        ]
    }
]

LANGUAGES = ["english", "hindi", "spanish"]
N_SAMPLES = 40
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading dataset...")
data = []
for lang in LANGUAGES:
    ds = load_dataset("cardiffnlp/tweet_sentiment_multilingual", lang, trust_remote_code=True)
    samples = ds["test"].select(range(N_SAMPLES))

    for s in samples:
        data.append({
            "text": s["text"],
            "label": s["label"],
            "lang": lang
        })
print(f"Total samples: {len(data)}")

activation = None

def hook_fn(module, input, output):
    global activation
    if isinstance(output, tuple):
        activation = output[0]
    else:
        activation = output

def get_features(text, sae, current_model, current_tokenizer):
    global activation
    inputs = current_tokenizer(text, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        _ = current_model(**inputs)

    dense = activation
    with torch.no_grad():
        sparse = sae.encode(dense)

    return sparse.mean(dim=1).squeeze().cpu()

for config in MODELS_CONFIG:
    print(f"\nStarting pipeline for: {config['name']}")

    print(f"Loading {config['model_id']}...")
    tokenizer = AutoTokenizer.from_pretrained(config['model_id'])

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        config['model_id'],
        torch_dtype=torch.float16 if "qwen" in config['model_id'].lower() else torch.float32,
        device_map="auto" if "qwen" in config['model_id'].lower() else None
    )

    if "gpt2" in config['model_id'].lower():
        model = model.to(DEVICE)

    def predict(text):
        prompt = f"Classify sentiment as Positive, Neutral, or Negative.\n\nTweet: {text}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

        resp = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).lower()

        if "positive" in resp: return 2
        if "neutral" in resp: return 1
        if "negative" in resp: return 0
        return -1

    print("Running zero-shot evaluation...")
    lang_acc = defaultdict(lambda: [0, 0])

    for d in data:
        p = predict(d["text"])
        lang_acc[d["lang"]][1] += 1
        if p == d["label"]:
            lang_acc[d["lang"]][0] += 1

    print(f"Baseline accuracy ({config['name']}):")
    for lang, (c, t) in lang_acc.items():
        print(f"{lang}: {c/t:.2%}")

    for sae_id in config['sae_ids']:
        layer = int(sae_id.split('.')[1])

        print(f"\nAnalyzing layer {layer} ({sae_id})")

        try:
            sae = SAE.from_pretrained(
                release=config['sae_release'],
                sae_id=sae_id,
                device=DEVICE
            )

            if "gpt2" in config['model_id'].lower():
                layer_module = model.transformer.h[layer]
            else:
                layer_module = model.model.layers[layer]

            hook = layer_module.register_forward_hook(hook_fn)

            feat_by_sent = defaultdict(list)
            feat_by_lang = defaultdict(list)

            for d in data:
                feat = get_features(d["text"], sae, model, tokenizer)
                feat_by_sent[d["label"]].append(feat)
                feat_by_lang[d["lang"]].append(feat)

            avg_sent = {k: torch.stack(v).mean(0) for k, v in feat_by_sent.items()}
            avg_lang = {k: torch.stack(v).mean(0) for k, v in feat_by_lang.items()}

            print("Sentiment features:")
            for k, v in avg_sent.items():
                print(f"Sentiment {k}: {(v > 0).sum().item()} active neurons")

            print("Language similarity (cosine):")
            langs = list(avg_lang.keys())
            for i in range(len(langs)):
                for j in range(i+1, len(langs)):
                    sim = torch.cosine_similarity(avg_lang[langs[i]], avg_lang[langs[j]], dim=0)
                    print(f"{langs[i]} vs {langs[j]}: {sim.item():.4f}")

            hook.remove()
            del sae

        except Exception as e:
            print(f"Could not process {sae_id}. Skipping...")
            print(f"Error detail: {e}")

        gc.collect()
        torch.cuda.empty_cache()

    print(f"Cleaning up {config['name']} from memory...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll pipelines complete.")

Loading dataset...
Total samples: 120

Starting pipeline for: Qwen 1.5B
Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Running zero-shot evaluation...
Baseline accuracy (Qwen 1.5B):
english: 67.50%
hindi: 35.00%
spanish: 62.50%

Analyzing layer 1 (blocks.1.hook_resid_pre)
Sentiment features:
Sentiment 0: 8324 active neurons
Sentiment 1: 7801 active neurons
Sentiment 2: 8220 active neurons
Language similarity (cosine):
english vs hindi: 0.9996
english vs spanish: 0.9992
hindi vs spanish: 0.9990
Cleaning up Qwen 1.5B from memory...

Starting pipeline for: GPT-2 Small
Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running zero-shot evaluation...
Baseline accuracy (GPT-2 Small):
english: 0.00%
hindi: 0.00%
spanish: 2.50%

Analyzing layer 0 (blocks.0.hook_resid_pre)
Sentiment features:
Sentiment 0: 24552 active neurons
Sentiment 1: 24549 active neurons
Sentiment 2: 24556 active neurons
Language similarity (cosine):
english vs hindi: 0.9823
english vs spanish: 0.9777
hindi vs spanish: 0.9761

Analyzing layer 6 (blocks.6.hook_resid_pre)
Sentiment features:
Sentiment 0: 24363 active neurons
Sentiment 1: 24361 active neurons
Sentiment 2: 24386 active neurons
Language similarity (cosine):
english vs hindi: 1.0000
english vs spanish: 1.0000
hindi vs spanish: 1.0000

Analyzing layer 11 (blocks.11.hook_resid_pre)
Sentiment features:
Sentiment 0: 24215 active neurons
Sentiment 1: 24020 active neurons
Sentiment 2: 24242 active neurons
Language similarity (cosine):
english vs hindi: 0.9876
english vs spanish: 0.9872
hindi vs spanish: 0.9802
Cleaning up GPT-2 Small from memory...

All pipelines complete.


In [7]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import gc
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

MODELS_CONFIG = [
    {
        "name": "Qwen 1.5B",
        "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "sae_release": "DGurgurov/DeepSeek-R1-Distill-Qwen-1.5B-sae",
        "sae_ids": [
            "blocks.1.hook_resid_pre"
        ]
    },
    {
        "name": "Qwen 7B",
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "sae_release": "rufimelo/secure_code_qwen_coder_strd_16384",
        "sae_ids": [
            "blocks.0.hook_resid_post",
            "blocks.14.hook_resid_post"
        ]
    }
]

LANGUAGES = ["english", "hindi", "spanish"]
N_SAMPLES = 40
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading dataset...")
data = []
for lang in LANGUAGES:
    ds = load_dataset("cardiffnlp/tweet_sentiment_multilingual", lang, trust_remote_code=True)
    samples = ds["test"].select(range(N_SAMPLES))

    for s in samples:
        data.append({
            "text": s["text"],
            "label": s["label"],
            "lang": lang
        })
print(f"Total samples: {len(data)}")

activation = None

def hook_fn(module, input, output):
    global activation
    if isinstance(output, tuple):
        activation = output[0]
    else:
        activation = output

def get_features(text, sae, current_model, current_tokenizer):
    global activation
    inputs = current_tokenizer(text, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        _ = current_model(**inputs)

    dense = activation
    with torch.no_grad():
        sparse = sae.encode(dense)

    return sparse.mean(dim=1).squeeze().cpu()

for config in MODELS_CONFIG:
    print(f"\nStarting pipeline for: {config['name']}")

    print(f"Loading {config['model_id']}...")
    tokenizer = AutoTokenizer.from_pretrained(config['model_id'])

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        config['model_id'],
        torch_dtype=torch.float16,
        device_map="auto"
    )

    def predict(text):
        prompt = f"Classify sentiment as Positive, Neutral, or Negative.\n\nTweet: {text}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

        resp = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).lower()

        if "positive" in resp: return 2
        if "neutral" in resp: return 1
        if "negative" in resp: return 0
        return -1

    print("Running zero-shot evaluation...")
    lang_acc = defaultdict(lambda: [0, 0])

    for d in data:
        p = predict(d["text"])
        lang_acc[d["lang"]][1] += 1
        if p == d["label"]:
            lang_acc[d["lang"]][0] += 1

    print(f"Baseline accuracy ({config['name']}):")
    for lang, (c, t) in lang_acc.items():
        print(f"{lang}: {c/t:.2%}")

    for sae_id in config['sae_ids']:
        layer = int(sae_id.split('.')[1])

        print(f"\nAnalyzing layer {layer} ({sae_id})")

        try:
            sae = SAE.from_pretrained(
                release=config['sae_release'],
                sae_id=sae_id,
                device=DEVICE
            )

            # Simplified hook registration since both models are now Qwen architecture
            layer_module = model.model.layers[layer]

            hook = layer_module.register_forward_hook(hook_fn)

            feat_by_sent = defaultdict(list)
            feat_by_lang = defaultdict(list)

            for d in data:
                feat = get_features(d["text"], sae, model, tokenizer)
                feat_by_sent[d["label"]].append(feat)
                feat_by_lang[d["lang"]].append(feat)

            avg_sent = {k: torch.stack(v).mean(0) for k, v in feat_by_sent.items()}
            avg_lang = {k: torch.stack(v).mean(0) for k, v in feat_by_lang.items()}

            print("Sentiment features:")
            for k, v in avg_sent.items():
                print(f"Sentiment {k}: {(v > 0).sum().item()} active neurons")

            print("Language similarity (cosine):")
            langs = list(avg_lang.keys())
            for i in range(len(langs)):
                for j in range(i+1, len(langs)):
                    sim = torch.cosine_similarity(avg_lang[langs[i]], avg_lang[langs[j]], dim=0)
                    print(f"{langs[i]} vs {langs[j]}: {sim.item():.4f}")

            hook.remove()
            del sae

        except Exception as e:
            print(f"Could not process {sae_id}. Skipping...")
            print(f"Error detail: {e}")

        gc.collect()
        torch.cuda.empty_cache()

    print(f"Cleaning up {config['name']} from memory...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll pipelines complete.")

Loading dataset...
Total samples: 120

Starting pipeline for: Qwen 1.5B
Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Running zero-shot evaluation...
Baseline accuracy (Qwen 1.5B):
english: 65.00%
hindi: 32.50%
spanish: 65.00%

Analyzing layer 1 (blocks.1.hook_resid_pre)
Sentiment features:
Sentiment 0: 8324 active neurons
Sentiment 1: 7801 active neurons
Sentiment 2: 8220 active neurons
Language similarity (cosine):
english vs hindi: 0.9996
english vs spanish: 0.9992
hindi vs spanish: 0.9990
Cleaning up Qwen 1.5B from memory...

Starting pipeline for: Qwen 7B
Loading Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Sentiment features:
Sentiment 0: 6174 active neurons
Sentiment 1: 5839 active neurons
Sentiment 2: 6728 active neurons
Language similarity (cosine):
english vs hindi: 0.4618
english vs spanish: 0.6378
hindi vs spanish: 0.4221

Analyzing layer 14 (blocks.14.hook_resid_post)


cfg.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

blocks.14.hook_resid_post/sae_weights.sa(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

blocks.14.hook_resid_post/sparsity.safet(…):   0%|          | 0.00/65.6k [00:00<?, ?B/s]

Sentiment features:
Sentiment 0: 16384 active neurons
Sentiment 1: 16384 active neurons
Sentiment 2: 16384 active neurons
Language similarity (cosine):
english vs hindi: 0.9228
english vs spanish: 0.9500
hindi vs spanish: 0.9192
Cleaning up Qwen 7B from memory...

All pipelines complete.
